# Integrating Vision & Voice
A senior AI Architect doesn't just build chatbots; they build **Experience Engines**. Today, we expand our agent's neural network to handle generative outputs:
1. **Vision :** Converting conceptual descriptions into high-fidelity visual assets.
2. **Voice :** Converting text strings into natural-sounding human speech.

We will learn how to manage the binary payloads (images and audio files) produced by these APIs and pipe them directly into our Gradio interface.

In [1]:
import os
import base64
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, Audio
import requests
from pathlib import Path

load_dotenv(override=True)
client = OpenAI()

print("🎨 Voice & Vision Environment Ready!")

🎨 Voice & Vision Environment Ready!


## 1. Architecting Vision:

As an Architect, you need to decide if you want the model to output a URL (temporary) or raw bytes (permanent).

In [2]:
def generate_architectural_diagram(prompt):
        """
        Generates a technical architectural diagram using gpt-image-1.
        """
        print(f"🖼️ [System] Generating architectural diagram for: {prompt}...")
        
        response = client.images.generate(
            model="gpt-image-1",
            prompt=f"A detailed, professional architectural blueprint and 3D diagram of: {prompt}. High resolution, technical style.",
        )

        image_base64 = response.data[0].b64_json

        if not image_base64:
            print("⚠️ [Warning] No image returned. Possibly blocked or failed.")
            return None

        # Save image locally
        image_bytes = base64.b64decode(image_base64)
        file_path = "architectural_diagram.png"
        
        with open(file_path, "wb") as f:
            f.write(image_bytes)

        print(f"✅ Diagram saved at: {file_path}")
        return file_path

test_path = generate_architectural_diagram("A sustainable glass-domed eco-research center")
print(f"Image Path: {test_path}")

🖼️ [System] Generating architectural diagram for: A sustainable glass-domed eco-research center...
✅ Diagram saved at: architectural_diagram.png
Image Path: architectural_diagram.png


## 2. Architecting Voice: OpenAI TTS

OpenAI's TTS is exceptionally fast. It returns binary data (usually MP3) which we must save to a temporary file before the UI can play it.

In [3]:
def text_to_speech(text, voice="onyx"):
    """
    Converts text to an audio file and returns the path.
    Voices: alloy, echo, fable, onyx, nova, shimmer
    """
    speech_file_path = Path("temp_speech.mp3")
    print(f"🔊 [System] Synthesizing voice ({voice})...")
    
    response = client.audio.speech.create(
        model="tts-1",
        voice=voice,
        input=text
    )
    
    # stream_to_file is deprecated in newer SDKs, we use with_streaming_response or standard write
    response.stream_to_file(speech_file_path)
    return str(speech_file_path)

test_audio = text_to_speech("Welcome to the multimodal era of AI engineering.")
Audio(test_audio)

🔊 [System] Synthesizing voice (onyx)...


/var/folders/rg/b21vyw1s0v727y0fc2phllm00000gn/T/ipykernel_11297/1254508971.py:16: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(speech_file_path)


## 3. The Creative Architect Agent

Now we build a unified function that takes a text prompt and returns **Text, Image, and Audio** simultaneously. This is the hallmark of a high-end AI experience.

In [4]:
import gradio as gr

def creative_architect_engine(user_prompt):
    # 1. Generate text response
    text_res = f"Architectural Brief: {user_prompt}. I have designed the visual layout and prepared the audio walkthrough."
    
    # 2. Generate Image
    img_url = generate_architectural_diagram(user_prompt)
    
    # 3. Generate Speech
    audio_path = text_to_speech(text_res)
    
    return text_res, img_url, audio_path

with gr.Blocks(theme=gr.themes.Monochrome()) as creative_suite:
    gr.Markdown("# Creative AI Architect v1")
    
    with gr.Row():
        with gr.Column():
            concept_input = gr.Textbox(label="Describe your Concept", placeholder="E.g., A futuristic airport terminal...")
            create_btn = gr.Button("Build Experience", variant="primary")
        
        with gr.Column():
            brief_out = gr.Textbox(label="System Brief")
            audio_out = gr.Audio(label="Audio Walkthrough")
            
    visual_out = gr.Image(label="Visual Blueprint")

    create_btn.click(
        fn=creative_architect_engine, 
        inputs=concept_input, 
        outputs=[brief_out, visual_out, audio_out]
    )

print("🚀 Creative Suite Ready.")
creative_suite.launch()

/var/folders/rg/b21vyw1s0v727y0fc2phllm00000gn/T/ipykernel_11297/3372684424.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as creative_suite:


🚀 Creative Suite Ready.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


🖼️ [System] Generating architectural diagram for: Create a 3 BHK House Layout...
✅ Diagram saved at: architectural_diagram.png
🔊 [System] Synthesizing voice (onyx)...


/var/folders/rg/b21vyw1s0v727y0fc2phllm00000gn/T/ipykernel_11297/1254508971.py:16: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(speech_file_path)
